In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Ensure timestamp is integer
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')

# helper to convert epoch-second min/max to Asia/Shanghai tz strings
def epoch_series_to_shanghai_str(s):
    # s is series of integer epoch seconds
    return pd.to_datetime(s.astype('int64'), unit='s', utc=True).dt.tz_convert('Asia/Shanghai').dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 1) metric.csv grouping
metrics_agg = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('value', 'count'),
    ts_min=('timestamp', 'min'),
    ts_max=('timestamp', 'max'),
    p50=('value', lambda x: x.quantile(0.5)),
    p90=('value', lambda x: x.quantile(0.9)),
    p95=('value', lambda x: x.quantile(0.95))
).reset_index()

# convert timestamps to Asia/Shanghai strings
metrics_agg['ts_min'] = epoch_series_to_shanghai_str(metrics_agg['ts_min'])
metrics_agg['ts_max'] = epoch_series_to_shanghai_str(metrics_agg['ts_max'])
metrics_agg[['p50','p90','p95']] = metrics_agg[['p50','p90','p95']].astype(float).round(6)

metrics_top20 = metrics_agg.sort_values('count', ascending=False).head(20)[
    ['cmdb_id','kpi_name','count','ts_min','ts_max','p50','p90','p95']
]

# 2) trace.csv grouping
trace_agg = trace_df.groupby(['cmdb_id', 'trace_name']).agg(
    count=('value', 'count'),
    ts_min=('timestamp', 'min'),
    ts_max=('timestamp', 'max'),
    p50=('value', lambda x: x.quantile(0.5)),
    p90=('value', lambda x: x.quantile(0.9)),
    p95=('value', lambda x: x.quantile(0.95))
).reset_index()

trace_agg['ts_min'] = epoch_series_to_shanghai_str(trace_agg['ts_min'])
trace_agg['ts_max'] = epoch_series_to_shanghai_str(trace_agg['ts_max'])
trace_agg[['p50','p90','p95']] = trace_agg[['p50','p90','p95']].astype(float).round(6)

trace_top20 = trace_agg.sort_values('count', ascending=False).head(20)[
    ['cmdb_id','trace_name','count','ts_min','ts_max','p50','p90','p95']
]

# 3) log.csv grouping: ensure numeric conversion for value percentiles
log_df['value_numeric'] = pd.to_numeric(log_df['value'], errors='coerce')

log_agg = log_df.groupby(['cmdb_id', 'log_name']).agg(
    count=('value', 'count'),
    ts_min=('timestamp', 'min'),
    ts_max=('timestamp', 'max'),
    p95_value=('value_numeric', lambda x: x.quantile(0.95))
).reset_index()

log_agg['ts_min'] = epoch_series_to_shanghai_str(log_agg['ts_min'])
log_agg['ts_max'] = epoch_series_to_shanghai_str(log_agg['ts_max'])
log_agg['p95_value'] = log_agg['p95_value'].astype(float).round(6)

log_top20 = log_agg.sort_values('count', ascending=False).head(20)[
    ['cmdb_id','log_name','count','ts_min','ts_max','p95_value']
]

# 4) error_logs.csv summary
error_total_rows = int(error_df.shape[0])
unique_cmdb = error_df['cmdb_id'].dropna().unique()
unique_cmdb_sample = list(unique_cmdb[:20])  # max 20
error_examples = error_df[['timestamp','cmdb_id','message']].head(10).copy()
# convert example timestamps to Asia/Shanghai strings
if not error_examples.empty:
    error_examples['timestamp'] = pd.to_datetime(error_examples['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert('Asia/Shanghai').dt.strftime('%Y-%m-%d %H:%M:%S%z')

error_summary = pd.DataFrame({
    'total_rows': [error_total_rows],
    'unique_cmdb_count': [int(len(unique_cmdb))],
    'unique_cmdb_sample': [unique_cmdb_sample]
})

# Display compact results (top tables and error summary/examples)
metrics_top20, trace_top20, log_top20, error_summary, error_examples

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregates (full series, UTC+8):\n\n"
    "1) Metrics (top groups):\n"
    "- Dominated by Tomcat04: multiple KPI series each with count=31, time range 2021-03-23 06:30:00+0800 to 2021-03-23 07:00:00+0800.\n"
    "- Example metric highlights: Tomcat-Requests (RequestCount) p50=1627687.0, p90=1628121.0, p95=1628169.0; CurrentThread ~100 (p50/p90/p95=100); MaxThreads=500 (p50/p90/p95=500).\n\n"
    "2) Traces (top groups):\n"
    "- Top trace groups have count=30 and time range 2021-03-23 06:30:00+0800 to 2021-03-23 06:59:00+0800.\n"
    "- Notable examples: dockerB2 -> trace.to_MG02.row_count p50=14.0 p90=37.1 p95=38.55; dockerB2 -> duration_p95 p95≈0.083s; dockerA2 trace.self.row_count p50=182.0 p95=298.9; MG01 trace.self.duration_p95 p95≈0.1655s.\n\n"
    "3) Logs (top groups):\n"
    "- Several log groups at count=30 (Tomcat01–Tomcat04, apache01–apache02). Time range 2021-03-23 06:30:00+0800 to 2021-03-23 06:59:00+0800 for those.\n"
    "- log.row_count p95 values range roughly from ~47 to ~93 across components; log.error_count p95 = 0.0 for top groups (no recorded errors in those series). gc_pause_max appears only a few samples with p95 around 0.07–0.08s.\n\n"
    "4) Error logs:\n"
    "- error_logs.csv contains 0 rows (no error log entries). No cmdb_id values found and no example rows to show.\n\n"
    "Overall: The dataset covers a short window on 2021-03-23 (06:30–07:00+0800). Tomcat04 produces the most metric series samples here; traces are concentrated on dockerA*/dockerB* and MG01/MG02 interactions; log row counts are high but error counts are zero in the top groups. Error log file is empty."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count                    ts_min                    ts_max          p50          p90          p95
1584  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1585  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1586  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1587  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1588  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1589  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800        100.0        100.0        100.0
1590  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1591  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800        500.0        500.0        500.0
1576  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800    1627687.0    1628121.0    1628169.0
1577  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1578  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1579  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1580  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1581  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1582  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1583  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          0.0          0.0          0.0
1573  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800        138.0        138.0        138.0
1574  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800     230487.0     230487.0     230487.0
1575  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800  853312376.0  853342139.0  853344899.5
1560  Tomcat04         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800          3.0          3.0          3.0,       cmdb_id                     trace_name  count                    ts_min                    ts_max         p50         p90         p95
260  dockerB2        trace.to_MG02.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800   14.000000   37.100000   38.550000
259  dockerB2     trace.to_MG02.duration_p95     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.009050    0.019760    0.083232
258  dockerB2    trace.to_MG02.duration_mean     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.006889    0.018433    0.020548
225  dockerB1  trace.from_MG01.duration_mean     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.036016    0.100376    0.137846
226  dockerB1   trace.from_MG01.duration_p95     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.075450    0.549780    0.747930
211  dockerA2   trace.from_MG02.duration_p95     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.037250    0.124570    0.386650
212  dockerA2      trace.from_MG02.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800   14.000000   24.300000   31.950000
216  dockerA2       trace.self.duration_mean     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.000500    0.001793    0.002804
217  dockerA2        trace.self.duration_p95     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.001000    0.008000    0.008028
218  dockerA2           trace.self.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800  182.000000  274.400000  298.900000
210  dockerA2  trace.from_MG02.duration_mean     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.022292    0.038227    0.095766
195  dockerA1  trace.from_MG02.duration_mean     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.025398    0.035793    0.044799
196  dockerA1   trace.from_MG02.duration_p95     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.043750    0.104530    0.171540
197  dockerA1      trace.from_MG02.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800   14.500000   21.100000   22.550000
198  dockerA1       trace.self.duration_mean     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.000720    0.002339    0.002605
199  dockerA1        trace.self.duration_p95     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.001025    0.007235    0.008550
200  dockerA1           trace.self.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800  175.000000  265.000000  275.100000
59       MG01  trace.from_dockerB2.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800   13.000000   24.400000   34.600000
63       MG01       trace.self.duration_mean     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.027338    0.053991    0.064954
64       MG01        trace.self.duration_p95     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    0.072200    0.130305    0.165547,      cmdb_id          log_name  count                    ts_min                    ts_max  p95_value
0   Tomcat01   log.error_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800     0.0000
2   Tomcat01     log.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    54.5500
3   Tomcat02   log.error_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800     0.0000
5   Tomcat02     log.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    48.3000
6   Tomcat03   log.error_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800     0.0000
11  Tomcat04     log.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    47.0000
9   Tomcat04   log.error_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800     0.0000
8   Tomcat03     log.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    53.7500
14  apache02   log.error_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800     0.0000
13  apache01     log.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    90.1000
15  apache02     log.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800    93.7500
12  apache01   log.error_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800     0.0000
10  Tomcat04  log.gc_pause_max      4  2021-03-23 06:30:00+0800  2021-03-23 06:58:00+0800     0.0785
4   Tomcat02  log.gc_pause_max      2  2021-03-23 06:42:00+0800  2021-03-23 06:59:00+0800     0.0690
7   Tomcat03  log.gc_pause_max      1  2021-03-23 06:45:00+0800  2021-03-23 06:45:00+0800     0.0700
1   Tomcat01  log.gc_pause_max      1  2021-03-23 06:43:00+0800  2021-03-23 06:43:00+0800     0.0800,    total_rows  unique_cmdb_count unique_cmdb_sample
0           0                  0                 [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df if present, else load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# ensure timestamp and value types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) global P95 per (cmdb_id, kpi_name) using full series
global_p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'] \
    .quantile(0.95) \
    .reset_index(name='global_p95')

# 2) filter metric.csv to the given UTC+8 window (inclusive)
tz_sh = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2021-03-23 06:30:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 07:00:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) find anomaly rows where value >= global_p95 for that series
metric_window = metric_window.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')
anomalies = metric_window[metric_window['value'] >= metric_window['global_p95']].copy()

# 4) aggregate anomaly rows by cmdb_id and kpi_name
if not anomalies.empty:
    agg = anomalies.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
        anomaly_count=('value', 'count'),
        earliest_anomaly_ts=('timestamp', 'min'),
        latest_anomaly_ts=('timestamp', 'max'),
        max_value_in_window=('value', 'max')
    ).reset_index()

    # attach global_p95 (same per group)
    agg = agg.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

    # compute severity_ratio safely (avoid divide by zero)
    agg['severity_ratio'] = np.where(
        agg['global_p95'] == 0,
        np.nan,
        agg['max_value_in_window'] / agg['global_p95']
    )

    # convert timestamps to Asia/Shanghai strings
    agg['earliest_anomaly_ts'] = pd.to_datetime(agg['earliest_anomaly_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    agg['latest_anomaly_ts'] = pd.to_datetime(agg['latest_anomaly_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # rounding numeric columns for compact display
    agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
    agg['global_p95'] = agg['global_p95'].round(6)
    agg['severity_ratio'] = agg['severity_ratio'].round(6)

    anomalies_summary_top20 = agg.sort_values('anomaly_count', ascending=False).head(20)[
        ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value_in_window','global_p95','severity_ratio']
    ]
else:
    # empty result placeholders
    anomalies_summary_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value_in_window','global_p95','severity_ratio'])

# 5) total number of anomaly rows
total_anomaly_rows = int(anomalies.shape[0])

# return only the compact aggregate table (top 20) and the total anomaly row count
anomalies_summary_top20, total_anomaly_rows

```
Out[2]:
```
summary = (
    "Anomaly summary (based on full-series P95 thresholds and window 2021-03-23 06:30:00 to 2021-03-23 07:00:00 UTC+8):\n\n"
    "- Total anomaly rows found: 17,825.\n\n"
    "- Top anomaly groups (most are Tomcat04 series, each with anomaly_count=31 covering 2021-03-23 06:30:00+0800 to 2021-03-23 07:00:00+0800):\n"
    "  * Tomcat-Threads CurrentThreads: max_in_window=100.0, global_p95=100.0, severity_ratio=1.0\n"
    "  * Tomcat-Threads MaxThreads: max_in_window=500.0, global_p95=500.0, severity_ratio=1.0\n"
    "  * Tomcat-Requests ErrorCount: max_in_window=138.0, global_p95=138.0, severity_ratio=1.0\n"
    "  * Tomcat-Requests MaxTimeRequest: max_in_window=230487.0, global_p95=230487.0, severity_ratio=1.0\n"
    "  * OSLinux process/network KPIs (examples) also show max==global_p95 and severity_ratio=1.0\n\n"
    "- Note: Several session-related KPIs for Tomcat04 have values and global_p95 equal to 0.0; severity_ratio is NaN in those cases due to division by zero (global_p95==0).\n\n"
    "Conclusion: Most anomalies are concentrated on Tomcat04 KPI series within the specified 30-minute window. Many series hit their historical P95 exactly (severity_ratio≈1.0). Some session metrics are zero-valued, producing undefined severity_ratio."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value_in_window  global_p95  severity_ratio
1584  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1585  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1586  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1587  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1588  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1589  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                100.0       100.0             1.0
1590  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  1.0         0.0             NaN
1591  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                500.0       500.0             1.0
1577  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1578  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1579  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1580  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1581  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1582  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1583  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN
1573  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                138.0       138.0             1.0
1574  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800             230487.0    230487.0             1.0
1560  Tomcat04         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  3.0         3.0             1.0
1556  Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  1.0         1.0             1.0
1545  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                  0.0         0.0             NaN, 17825)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df if present, else load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (if not already computed)
if 'global_p95_df' not in globals():
    global_p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'] \
        .quantile(0.95) \
        .reset_index(name='global_p95')

# 2) Filter metric.csv to the incident window 2021-03-23 06:30:00 to 2021-03-23 07:00:00 (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2021-03-23 06:30:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 07:00:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Find rows where value >= global_p95 for that (cmdb_id, kpi_name)
metric_window = metric_window.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Keep only rows where global_p95 is numeric (non-null) and value >= global_p95
candidate_rows = metric_window[metric_window['global_p95'].notnull() & (metric_window['value'] >= metric_window['global_p95'])].copy()

# If no candidate rows, prepare empty outputs
if candidate_rows.empty:
    faults_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_ts','fault_end_ts','duration_minutes',
        'max_value_in_run','global_p95','severity_ratio'
    ])
    total_faults = 0
else:
    # 3 continued) Identify consecutive timestamp runs (diff == 60 sec within each series)
    candidate_rows = candidate_rows.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)
    # compute diff within each (cmdb_id,kpi_name)
    candidate_rows['ts_diff'] = candidate_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].diff().fillna(999999)
    # start of a new run when ts_diff != 60
    candidate_rows['new_run'] = (candidate_rows['ts_diff'] != 60).astype(int)
    # run_number per series
    candidate_rows['run_number'] = candidate_rows.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum()
    # create a run id combining keys
    candidate_rows['run_id'] = candidate_rows['cmdb_id'].astype(str) + '||' + candidate_rows['kpi_name'].astype(str) + '||' + candidate_rows['run_number'].astype(str)

    # 4) Aggregate runs; keep only runs with length >= 2 (duration_minutes >=2)
    runs = candidate_rows.groupby(['cmdb_id','kpi_name','run_id'], dropna=False).agg(
        fault_start_ts=('timestamp','min'),
        fault_end_ts=('timestamp','max'),
        duration_minutes=('timestamp','count'),
        max_value_in_run=('value','max'),
        global_p95=('global_p95','first')
    ).reset_index(drop=False)

    # filter duration >= 2
    faults = runs[runs['duration_minutes'] >= 2].copy()

    # compute severity_ratio safely
    faults['severity_ratio'] = np.where(
        faults['global_p95'] == 0,
        np.nan,
        faults['max_value_in_run'] / faults['global_p95']
    )

    # convert timestamps to Asia/Shanghai strings
    faults['fault_start_ts'] = pd.to_datetime(faults['fault_start_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    faults['fault_end_ts'] = pd.to_datetime(faults['fault_end_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # rounding numeric columns
    faults['max_value_in_run'] = faults['max_value_in_run'].round(6)
    faults['global_p95'] = faults['global_p95'].round(6)
    faults['severity_ratio'] = faults['severity_ratio'].round(6)

    # 5) Return compact table sorted by duration_minutes desc then severity_ratio desc, top 20 rows
    faults_top20 = faults.sort_values(['duration_minutes','severity_ratio'], ascending=[False, False]) \
        .head(20)[['cmdb_id','kpi_name','fault_start_ts','fault_end_ts','duration_minutes','max_value_in_run','global_p95','severity_ratio']].reset_index(drop=True)

    # 6) total number of faults detected
    total_faults = int(faults.shape[0])

# Display only the compact results
faults_top20, total_faults

```
Out[3]:
```
summary = (
    "Fault detection summary (window 2021-03-23 06:30:00 to 2021-03-23 07:00:00 UTC+8):\n\n"
    "• Total faults detected: 1514\n\n"
    "• Top detected faults (sorted by duration then severity):\n"
    "  1) MG02 — OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P... | 2021-03-23 06:30 → 07:00 | 31 min | max=0.4 | global_p95=0.3 | severity=1.333\n"
    "  2) Tomcat02 — OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P... | 2021-03-23 06:30 → 07:00 | 31 min | max=0.4 | global_p95=0.3 | severity=1.333\n"
    "  3) Mysql02 — Mysql-MySQL_3306_ThreadsConnected | 2021-03-23 06:30 → 07:00 | 31 min | max=21.0 | global_p95=20.0 | severity=1.05\n"
    "  4) IG01 — JVM-Memory_7778_JVM_Memory_HeapMemoryMax | 2021-03-23 06:30 → 07:00 | 31 min | max=2.040136e+09 | global_p95=2.040136e+09 | severity=1.00\n"
    "  5) IG01 — JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed | 2021-03-23 06:30 → 07:00 | 31 min | max=5.805683e+07 | global_p95=5.805683e+07 | severity=1.00\n"
    "  6) IG02 / MG01 / MG02 / Mysql01 — multiple JVM, OS, and MySQL KPIs similarly show 31-minute continuous runs with max ≈ global_p95 (severity ≈ 1.0).\n\n"
    "• Observations:\n"
    "  - Most detected faults are continuous runs spanning the entire 31-minute window (duration_minutes = 31).\n"
    "  - The largest relative breach among the top faults is severity ≈ 1.333 (zabbix agent process metric on MG02/Tomcat02).\n"
    "  - Many JVM and DB KPIs simply reached their historical P95 (severity ≈ 1.0), indicating sustained high but not extreme spikes beyond the historical 95th percentile.\n\n"
    "Conclusion: The incident window shows widespread, sustained P95-level behavior across OS, JVM, and DB KPIs (many continuous 31-minute runs). The most notable above-threshold severity is the zabbix-agentd process metric (≈33% above its global P95)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name            fault_start_ts              fault_end_ts  duration_minutes  max_value_in_run    global_p95  severity_ratio
0       MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      4.000000e-01  3.000000e-01        1.333333
1   Tomcat02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      4.000000e-01  3.000000e-01        1.333333
2    Mysql02                  Mysql-MySQL_3306_ThreadsConnected  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      2.100000e+01  2.000000e+01        1.050000
3       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      2.040136e+09  2.040136e+09        1.000000
4       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      5.805683e+07  5.805683e+07        1.000000
5       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      2.040136e+09  2.040136e+09        1.000000
6       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      5.811566e+07  5.811566e+07        1.000000
7       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      2.040136e+09  2.040136e+09        1.000000
8       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      5.894522e+07  5.894522e+07        1.000000
9       MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      2.000000e+00  2.000000e+00        1.000000
10      MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      3.000000e-01  3.000000e-01        1.000000
11      MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      3.000000e+00  3.000000e+00        1.000000
12      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      2.040136e+09  2.040136e+09        1.000000
13      MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      1.000000e+00  1.000000e+00        1.000000
14      MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      3.000000e+00  3.000000e+00        1.000000
15   Mysql01             Mysql-MySQL_3306_Binlog stmt cache use  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      3.600000e+02  3.600000e+02        1.000000
16   Mysql01     Mysql-MySQL_3306_Innodb buffer pool pages free  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      2.522180e+05  2.522180e+05        1.000000
17   Mysql01            Mysql-MySQL_3306_Innodb open files  num  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      1.650000e+02  1.650000e+02        1.000000
18   Mysql01          Mysql-MySQL_3306_Innodb row lock time avg  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      3.210000e+02  3.210000e+02        1.000000
19   Mysql01          Mysql-MySQL_3306_Innodb row lock time max  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800                31      3.186700e+04  3.186700e+04        1.000000, 1514)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse existing trace_df if available, else load
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")

# ensure types
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) compute global P95 per (cmdb_id, trace_name) using full series
trace_global_p95 = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'] \
    .quantile(0.95) \
    .reset_index(name='global_p95')

# 2) filter trace_df to incident window (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2021-03-23 06:30:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 07:00:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())

trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) find anomaly rows where value >= global_p95 (join thresholds computed from full series)
trace_window = trace_window.merge(trace_global_p95, on=['cmdb_id','trace_name'], how='left')
trace_anoms = trace_window[trace_window['global_p95'].notnull() & (trace_window['value'] >= trace_window['global_p95'])].copy()

# 4) aggregate anomaly rows by cmdb_id and trace_name
if trace_anoms.empty:
    trace_anom_top20 = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
        'max_value_in_window','global_p95','severity_ratio'
    ])
    total_trace_anomaly_rows = 0
else:
    agg = trace_anoms.groupby(['cmdb_id','trace_name'], dropna=False).agg(
        anomaly_count=('value','count'),
        earliest_anomaly_ts=('timestamp','min'),
        latest_anomaly_ts=('timestamp','max'),
        max_value_in_window=('value','max'),
        global_p95=('global_p95','first')
    ).reset_index()

    # severity ratio, handle global_p95==0
    agg['severity_ratio'] = np.where(agg['global_p95']==0, np.nan, agg['max_value_in_window']/agg['global_p95'])

    # convert timestamps to Asia/Shanghai strings
    agg['earliest_anomaly_ts'] = pd.to_datetime(agg['earliest_anomaly_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    agg['latest_anomaly_ts'] = pd.to_datetime(agg['latest_anomaly_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # round numeric columns
    agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
    agg['global_p95'] = agg['global_p95'].round(6)
    agg['severity_ratio'] = agg['severity_ratio'].round(6)

    # 5) top 20 by anomaly_count desc
    trace_anom_top20 = agg.sort_values('anomaly_count', ascending=False).head(20)[
        ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value_in_window','global_p95','severity_ratio']
    ].reset_index(drop=True)

    # 6) total number of trace anomaly rows
    total_trace_anomaly_rows = int(trace_anoms.shape[0])

# return compact results
trace_anom_top20, total_trace_anomaly_rows

```
Out[4]:
```
summary = (
    "Trace anomaly summary (thresholds = global P95 computed from full series; window = 2021-03-23 06:30–07:00 UTC+8):\n\n"
    "• Total trace anomaly rows found in the window: 505\n\n"
    "• Top anomaly groups (compact highlights):\n"
    "  - dockerB1 / trace.self.duration_p95: 4 anomalies, 2021-03-23 06:45 → 06:56, max=0.01180, global_p95=0.01100, severity≈1.073\n"
    "  - IG01 / trace.to_Tomcat01.row_count: 4 anomalies, 06:38 → 06:53, max=26, global_p95=25, severity=1.04\n"
    "  - Tomcat01 / trace.from_IG01.row_count: 4 anomalies, 06:38 → 06:53, max=26, global_p95=25, severity=1.04\n"
    "  - dockerA1 / trace.to_MG02.row_count: 3 anomalies, 06:35 → 06:51, max=11, global_p95=8, severity=1.375\n"
    "  - MG02 / trace.from_Tomcat01.row_count: 3 anomalies, 06:42 → 06:46, max=43, global_p95=42, severity≈1.024\n"
    "  - Many other groups (dockerA2, dockerB2, Tomcat02/04, MG01/02, IG02) show 2–3 anomaly points—mostly row_count and duration metrics.\n\n"
    "• Notable high-relative breaches:\n"
    "  - dockerA2 / trace.to_MG02.duration_mean: max≈0.07575 vs global_p95≈0.01188 → severity ≈ 6.38 (large relative increase)\n"
    "  - dockerA2 / trace.to_MG02.duration_p95: max≈0.24005 vs global_p95≈0.01644 → severity ≈ 14.60 (very large relative increase)\n\n"
    "• Interpretation:\n"
    "  - Anomalies are concentrated on inter-service trace row_count and duration metrics among docker*/MG/IG/Tomcat components.\n"
    "  - Most groups show modest exceedances (severity ~1.02–1.46), but a few duration metrics on dockerA2 have substantial relative spikes and warrant deeper investigation.\n\n"
    "Recommendation: investigate dockerA2 → MG02 traces (duration_p95/duration_mean) first, then review the row_count bursts between IG/Tomcat and MG/docker components."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                     trace_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value_in_window  global_p95  severity_ratio
0   dockerB1        trace.self.duration_p95              4  2021-03-23 06:45:00+0800  2021-03-23 06:56:00+0800              0.01180    0.011000        1.072727
1       IG01    trace.to_Tomcat01.row_count              4  2021-03-23 06:38:00+0800  2021-03-23 06:53:00+0800             26.00000   25.000000        1.040000
2   Tomcat01      trace.from_IG01.row_count              4  2021-03-23 06:38:00+0800  2021-03-23 06:53:00+0800             26.00000   25.000000        1.040000
3   dockerA1        trace.to_MG02.row_count              3  2021-03-23 06:35:00+0800  2021-03-23 06:51:00+0800             11.00000    8.000000        1.375000
4       MG02  trace.from_Tomcat01.row_count              3  2021-03-23 06:42:00+0800  2021-03-23 06:46:00+0800             43.00000   42.000000        1.023810
5   Tomcat01        trace.to_MG02.row_count              3  2021-03-23 06:42:00+0800  2021-03-23 06:46:00+0800             43.00000   42.000000        1.023810
6       IG02    trace.to_Tomcat02.row_count              3  2021-03-23 06:44:00+0800  2021-03-23 06:57:00+0800             26.00000   25.000000        1.040000
7   dockerB2      trace.from_MG01.row_count              3  2021-03-23 06:34:00+0800  2021-03-23 06:40:00+0800             26.00000   25.000000        1.040000
8   Tomcat04      trace.from_IG02.row_count              3  2021-03-23 06:43:00+0800  2021-03-23 06:54:00+0800             26.00000   25.000000        1.040000
9       MG01    trace.to_dockerB2.row_count              3  2021-03-23 06:34:00+0800  2021-03-23 06:40:00+0800             26.00000   25.000000        1.040000
10      IG02    trace.to_Tomcat04.row_count              3  2021-03-23 06:43:00+0800  2021-03-23 06:54:00+0800             26.00000   25.000000        1.040000
11      MG02  trace.from_dockerA1.row_count              3  2021-03-23 06:35:00+0800  2021-03-23 06:51:00+0800             11.00000    8.000000        1.375000
12  Tomcat02      trace.from_IG02.row_count              3  2021-03-23 06:44:00+0800  2021-03-23 06:57:00+0800             26.00000   25.000000        1.040000
13  dockerA2     trace.to_MG01.duration_p95              2  2021-03-23 06:53:00+0800  2021-03-23 06:54:00+0800              0.01425    0.013200        1.079545
14  dockerA2    trace.to_MG02.duration_mean              2  2021-03-23 06:30:00+0800  2021-03-23 06:52:00+0800              0.07575    0.011875        6.378947
15  dockerA2        trace.to_MG01.row_count              2  2021-03-23 06:39:00+0800  2021-03-23 06:52:00+0800             13.00000    8.900000        1.460674
16  dockerA2    trace.to_MG01.duration_mean              2  2021-03-23 06:58:00+0800  2021-03-23 06:59:00+0800              0.01100    0.009967        1.103675
17  dockerB1  trace.from_MG01.duration_mean              2  2021-03-23 06:45:00+0800  2021-03-23 06:59:00+0800              0.17680    0.137846        1.282593
18  dockerB1   trace.from_MG01.duration_p95              2  2021-03-23 06:41:00+0800  2021-03-23 06:59:00+0800              0.81850    0.747930        1.094354
19  dockerA2     trace.to_MG02.duration_p95              2  2021-03-23 06:30:00+0800  2021-03-23 06:52:00+0800              0.24005    0.016438       14.603802, 505)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
import re

# reuse existing variables if present
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")

# ensure types
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) recompute global P95 per (cmdb_id, trace_name) using full series
trace_global_p95 = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'] \
    .quantile(0.95) \
    .reset_index(name='global_p95')

# 2) filter to incident window (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2021-03-23 06:30:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 07:00:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())

trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) find anomaly rows where value >= global_p95 (join thresholds computed from full series)
trace_window = trace_window.merge(trace_global_p95, on=['cmdb_id','trace_name'], how='left')
trace_anoms = trace_window[trace_window['global_p95'].notnull() & (trace_window['value'] >= trace_window['global_p95'])].copy()

# 4) parse trace_name for target components using patterns 'to_<COMP>' or 'from_<COMP>'
allowed_components = {'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
                      'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'}

# extract first occurrence of to_/from_ pattern
trace_anoms['matched_component'] = trace_anoms['trace_name'].str.extract(r'(?:to|from)_([A-Za-z0-9]+)', expand=False)

# keep only rows where matched_component is in allowed list
trace_anoms_matched = trace_anoms[trace_anoms['matched_component'].isin(allowed_components)].copy()

# 5) aggregate by matched_component
if trace_anoms_matched.empty:
    trace_component_agg_top20 = pd.DataFrame(columns=[
        'matched_component','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
        'max_value_in_window','global_p95_for_max','severity_ratio'
    ])
    total_matched_trace_anomalies = 0
else:
    # compute basic aggregates
    grouped = trace_anoms_matched.groupby('matched_component', dropna=False).agg(
        anomaly_count=('value','count'),
        earliest_anomaly_ts=('timestamp','min'),
        latest_anomaly_ts=('timestamp','max'),
        max_value_in_window=('value','max')
    ).reset_index()

    # for each matched_component, find the global_p95 corresponding to the row that produced max_value_in_window
    # merge to get global_p95 where value == max_value_in_window; if multiple, take first
    merged = grouped.merge(trace_anoms_matched[['matched_component','value','global_p95','timestamp']],
                           left_on=['matched_component','max_value_in_window'],
                           right_on=['matched_component','value'],
                           how='left', suffixes=('','_row'))

    # In some cases merge may produce multiple rows per group; keep first occurrence per matched_component
    merged = merged.sort_values(['matched_component','timestamp']).drop_duplicates('matched_component', keep='first')

    # rename and compute severity_ratio
    merged = merged.rename(columns={'global_p95':'global_p95_for_max'})
    merged['severity_ratio'] = np.where(merged['global_p95_for_max'] == 0, np.nan,
                                        merged['max_value_in_window'] / merged['global_p95_for_max'])

    # convert timestamps to Asia/Shanghai strings
    merged['earliest_anomaly_ts'] = pd.to_datetime(merged['earliest_anomaly_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    merged['latest_anomaly_ts'] = pd.to_datetime(merged['latest_anomaly_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # round numeric columns
    merged['max_value_in_window'] = merged['max_value_in_window'].round(6)
    merged['global_p95_for_max'] = merged['global_p95_for_max'].round(6)
    merged['severity_ratio'] = merged['severity_ratio'].round(6)

    # prepare final table and sort by anomaly_count desc, limit top 20
    trace_component_agg_top20 = merged.sort_values('anomaly_count', ascending=False).head(20)[
        ['matched_component','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
         'max_value_in_window','global_p95_for_max','severity_ratio']
    ].reset_index(drop=True)

    total_matched_trace_anomalies = int(trace_anoms_matched.shape[0])

# Return compact results
trace_component_agg_top20, total_matched_trace_anomalies

```
Out[5]:
```
summary = (
    "Trace anomaly component mapping (thresholds = global P95 from full series; window = 2021-03-23 06:30–07:00 UTC+8):\n\n"
    "• Total trace anomaly rows matched to allowed components: 300\n\n"
    "• Top implicated components (by matched anomaly_count):\n"
    "  - MG02: 74 anomalies, window 06:30–06:56, max=54.0 vs global_p95=41.90 → severity≈1.289\n"
    "  - MG01: 73 anomalies, window 06:31–06:59, max=47.0 vs global_p95=34.60 → severity≈1.358\n"
    "  - Tomcat01: 27 anomalies, window 06:38–06:54, max=43.0 vs global_p95=42.00 → severity≈1.024\n"
    "  - IG02: 26 anomalies, window 06:39–06:57, max=33.0 vs global_p95=30.65 → severity≈1.077\n"
    "  - IG01: 26 anomalies, window 06:31–06:53, max=29.0 vs global_p95=24.75 → severity≈1.172\n"
    "  - Tomcat02: 25 anomalies, window 06:31–06:59, max=47.0 vs global_p95=33.10 → severity≈1.420\n"
    "  - Tomcat04: 25 anomalies, window 06:31–06:54, max=42.0 vs global_p95=38.10 → severity≈1.102\n"
    "  - Tomcat03: 24 anomalies, window 06:38–06:58, max=54.0 vs global_p95=41.90 → severity≈1.289\n\n"
    "Interpretation and next steps:\n"
    "  - Most trace anomalies map to MG01 and MG02 (largest counts) and multiple Tomcat/IG components.\n"
    "  - Highest relative breach among the top groups is Tomcat02 (severity ≈ 1.42) and MG01 (≈1.36).\n"
    "  - Recommend investigating MG02/MG01 first (highest anomaly volume) and Tomcat02 (notable relative spike), focusing on the trace routes and services identified in the time window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(  matched_component  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value_in_window  global_p95_for_max  severity_ratio
0              MG02             74  2021-03-23 06:30:00+0800  2021-03-23 06:56:00+0800                 54.0               41.90        1.288783
1              MG01             73  2021-03-23 06:31:00+0800  2021-03-23 06:59:00+0800                 47.0               34.60        1.358382
2          Tomcat01             27  2021-03-23 06:38:00+0800  2021-03-23 06:54:00+0800                 43.0               42.00        1.023810
3              IG02             26  2021-03-23 06:39:00+0800  2021-03-23 06:57:00+0800                 33.0               30.65        1.076672
4              IG01             26  2021-03-23 06:31:00+0800  2021-03-23 06:53:00+0800                 29.0               24.75        1.171717
5          Tomcat02             25  2021-03-23 06:31:00+0800  2021-03-23 06:59:00+0800                 47.0               33.10        1.419940
6          Tomcat04             25  2021-03-23 06:31:00+0800  2021-03-23 06:54:00+0800                 42.0               38.10        1.102362
7          Tomcat03             24  2021-03-23 06:38:00+0800  2021-03-23 06:58:00+0800                 54.0               41.90        1.288783, 300)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df if present, else load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# ensure types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) compute global P95 per (cmdb_id, kpi_name) using full series (thresholds computed BEFORE any time filtering)
metric_global_p95 = metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'] \
    .quantile(0.95) \
    .reset_index(name='global_p95')

# 2) filter metric.csv to the incident window (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2021-03-23 06:30:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 07:00:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) restrict to top candidate components
candidates = {'MG02', 'MG01', 'Tomcat02'}
metric_window_candidates = metric_window[metric_window['cmdb_id'].isin(candidates)].copy()

# join thresholds (global_p95) computed from full series
metric_window_candidates = metric_window_candidates.merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')

# find anomaly rows where value >= global_p95 (and global_p95 is present)
anomalies = metric_window_candidates[metric_window_candidates['global_p95'].notnull() & (metric_window_candidates['value'] >= metric_window_candidates['global_p95'])].copy()

# total anomaly rows matched to the three components
total_anomaly_rows_matched = int(anomalies.shape[0])

# 4) aggregate anomalies per (cmdb_id, kpi_name)
if anomalies.empty:
    metric_anom_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
        'max_value_in_window','global_p95','severity_ratio'
    ])
else:
    agg = anomalies.groupby(['cmdb_id','kpi_name'], dropna=False).agg(
        anomaly_count=('value','count'),
        earliest_anomaly_ts=('timestamp','min'),
        latest_anomaly_ts=('timestamp','max'),
        max_value_in_window=('value','max'),
        global_p95=('global_p95','first')
    ).reset_index()

    # compute severity_ratio, handle global_p95 == 0
    agg['severity_ratio'] = np.where(agg['global_p95'] == 0, np.nan, agg['max_value_in_window'] / agg['global_p95'])

    # convert timestamps to Asia/Shanghai strings
    agg['earliest_anomaly_ts'] = pd.to_datetime(agg['earliest_anomaly_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    agg['latest_anomaly_ts'] = pd.to_datetime(agg['latest_anomaly_ts'].astype('int64'), unit='s', utc=True) \
        .dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # rounding numeric columns for compact display
    agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
    agg['global_p95'] = agg['global_p95'].round(6)
    agg['severity_ratio'] = agg['severity_ratio'].round(6)

    # 5) sort by severity_ratio desc then anomaly_count desc, limit to top 20
    metric_anom_top20 = agg.sort_values(['severity_ratio','anomaly_count'], ascending=[False, False]).head(20)[
        ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value_in_window','global_p95','severity_ratio']
    ].reset_index(drop=True)

# Return only the compact aggregate table and the single total count
metric_anom_top20, total_anomaly_rows_matched

```
Out[7]:
```
Summary of results (metrics for MG02, MG01, Tomcat02 within 2021-03-23 06:30→07:00 UTC+8):

- Total anomaly rows matched to MG02, MG01, Tomcat02: 3,117.

- Top anomalies by severity ratio (high → low):
  1. MG01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT: 2 anomalies (06:42 → 06:46), max=2.0 vs global_p95=0.5, severity=4.00.
  2. MG02 — OSLinux-CPU_CPU-3_SingleCpuidle: 1 anomaly (06:49), max≈2.5087 vs global_p95≈0.75261, severity≈3.33.
  3. MG02 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps: 1 anomaly (06:59), max=8 vs global_p95=3.1, severity≈2.58.
  4. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps: 1 anomaly (06:47), max=6 vs global_p95=2.5, severity=2.40.
  5. MG01 — OSLinux-CPU_CPU-2_SingleCpuidle: 1 anomaly (06:45), max≈98.95 vs global_p95≈47.39, severity≈2.09.
  6. (Others include disk I/O, network throughput, CPU WIO, and per-CPU utilizations with severity between ~1.3–1.7.)
  7. MG02 and Tomcat02 — zabbix-agentd process metric: continuous anomalies (31 samples each across the full window), max=0.4 vs global_p95=0.3, severity=1.333.

- Patterns and interpretation:
  - Anomalies are dominated by OS-level metrics (network, disk, CPU) on MG01 and MG02 and some CPU metrics on Tomcat02.
  - Some metrics show very large relative breaches (e.g., TCP FIN-WAIT on MG01, CPU idle on MG02), while many series repeatedly hit or slightly exceed their historical P95 (severity ≈ 1.0–1.4).
  - The zabbix-agentd process metric shows sustained exceedance across the whole 31-minute window on MG02 and Tomcat02 (31 samples).

- Recommended next steps:
  - Investigate MG01 and MG02 OS-level resource signals first (network TCP states, disk throughput, CPU per-core idle/utilization) since they have both high severity ratios and high anomaly volumes.
  - Check Tomcat02 CPU WIO and per-CPU utilization for potential contention.
  - Review persistent zabbix_agentd behavior (sustained P95 exceedance) to rule out monitoring-induced load or agent issues.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value_in_window     global_p95  severity_ratio
0       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              2  2021-03-23 06:42:00+0800  2021-03-23 06:46:00+0800             2.000000       0.500000        4.000000
1       MG02                    OSLinux-CPU_CPU-3_SingleCpuidle              1  2021-03-23 06:49:00+0800  2021-03-23 06:49:00+0800             2.508700       0.752610        3.333333
2       MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps              1  2021-03-23 06:59:00+0800  2021-03-23 06:59:00+0800             8.000000       3.100000        2.580645
3       MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps              1  2021-03-23 06:47:00+0800  2021-03-23 06:47:00+0800             6.000000       2.500000        2.400000
4       MG01                    OSLinux-CPU_CPU-2_SingleCpuidle              1  2021-03-23 06:45:00+0800  2021-03-23 06:45:00+0800            98.952800      47.392339        2.087949
5       MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps              1  2021-03-23 06:55:00+0800  2021-03-23 06:55:00+0800             4.950000       2.885000        1.715771
6       MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps              1  2021-03-23 06:45:00+0800  2021-03-23 06:45:00+0800             7.000000       4.200000        1.666667
7       MG01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec              1  2021-03-23 06:39:00+0800  2021-03-23 06:39:00+0800        172147.000000  106659.900000        1.613981
8   Tomcat02                             OSLinux-CPU_CPU_CPUWio              2  2021-03-23 06:43:00+0800  2021-03-23 06:56:00+0800             0.345057       0.214092        1.611727
9   Tomcat02                    OSLinux-CPU_CPU-2_SingleCpuUtil              1  2021-03-23 06:47:00+0800  2021-03-23 06:47:00+0800             8.623300       5.493200        1.569814
10      MG02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec              1  2021-03-23 06:43:00+0800  2021-03-23 06:43:00+0800        234747.000000  161562.850000        1.452976
11  Tomcat02                    OSLinux-CPU_CPU-0_SingleCpuUtil              1  2021-03-23 06:59:00+0800  2021-03-23 06:59:00+0800             3.129700       2.246700        1.393021
12      MG01                             OSLinux-CPU_CPU_CPUWio              2  2021-03-23 06:41:00+0800  2021-03-23 06:42:00+0800             0.615300       0.442649        1.390040
13      MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1  2021-03-23 06:32:00+0800  2021-03-23 06:32:00+0800             0.500000       0.360000        1.388889
14  Tomcat02    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil              1  2021-03-23 06:44:00+0800  2021-03-23 06:44:00+0800             0.000800       0.000590        1.355932
15      MG01    OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil              1  2021-03-23 06:51:00+0800  2021-03-23 06:51:00+0800             0.001100       0.000820        1.341463
16      MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800             0.400000       0.300000        1.333333
17  Tomcat02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-23 06:30:00+0800  2021-03-23 07:00:00+0800             0.400000       0.300000        1.333333
18      MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...              1  2021-03-23 06:43:00+0800  2021-03-23 06:43:00+0800             0.300000       0.230000        1.304348
19      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1  2021-03-23 06:55:00+0800  2021-03-23 06:55:00+0800             0.300000       0.230000        1.304348, 3117)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# reuse existing log_df if available, else load
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")

# ensure types
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
# create numeric copy of value for percentile calc
log_df['value_numeric'] = pd.to_numeric(log_df.get('value', pd.Series(dtype='float')), errors='coerce')

# define window in UTC based on Asia/Shanghai (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2021-03-23 06:30:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 07:00:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())

# filter rows to window and candidate components
candidates = {'MG01','MG02','Tomcat02'}
log_window = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts) & (log_df['cmdb_id'].isin(candidates))].copy()

# total matched rows
total_matched_log_rows = int(log_window.shape[0])

# if no rows, prepare empty result
if log_window.empty:
    log_agg_top20 = pd.DataFrame(columns=['cmdb_id','log_name','count','earliest_ts','latest_ts','p95_value'])
else:
    # group by cmdb_id and log_name
    agg = log_window.groupby(['cmdb_id','log_name'], dropna=False).agg(
        count=('timestamp','count'),
        earliest_ts=('timestamp','min'),
        latest_ts=('timestamp','max'),
        p95_value=('value_numeric', lambda x: x.quantile(0.95))
    ).reset_index()

    # convert timestamps to Asia/Shanghai formatted strings
    agg['earliest_ts'] = pd.to_datetime(agg['earliest_ts'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    agg['latest_ts'] = pd.to_datetime(agg['latest_ts'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # round p95_value for compactness; keep NaN if non-numeric
    agg['p95_value'] = agg['p95_value'].astype(float).round(6)

    # select and sort top 20 by count desc
    log_agg_top20 = agg.sort_values('count', ascending=False).head(20)[
        ['cmdb_id','log_name','count','earliest_ts','latest_ts','p95_value']
    ].reset_index(drop=True)

# return only the compact aggregate table and the single total count
log_agg_top20, total_matched_log_rows

```
Out[8]:
```
summary = (
    "Log summary for 2021-03-23 06:30 → 07:00 (UTC+8) restricted to MG01, MG02, Tomcat02:\n\n"
    "- Total log rows matched to these components in the window: 62\n\n"
    "- Top groups (by count):\n"
    "  * Tomcat02 — log.error_count: count=30, earliest=2021-03-23 06:30:00+0800, latest=2021-03-23 06:59:00+0800, p95_value=0.0 (no errors at 95th percentile)\n"
    "  * Tomcat02 — log.row_count:   count=30, earliest=2021-03-23 06:30:00+0800, latest=2021-03-23 06:59:00+0800, p95_value=48.3\n"
    "  * Tomcat02 — log.gc_pause_max: count=2,  earliest=2021-03-23 06:42:00+0800, latest=2021-03-23 06:59:00+0800, p95_value=0.069\n\n"
    "Interpretation: Tomcat02 produces all matched log activity in the window. Errors are not observed at the 95th-percentile (p95=0), row counts are substantial (p95≈48.3), and there are only a couple of GC pause samples with p95≈0.069s."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id          log_name  count               earliest_ts                 latest_ts  p95_value
0  Tomcat02   log.error_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800      0.000
1  Tomcat02     log.row_count     30  2021-03-23 06:30:00+0800  2021-03-23 06:59:00+0800     48.300
2  Tomcat02  log.gc_pause_max      2  2021-03-23 06:42:00+0800  2021-03-23 06:59:00+0800      0.069, 62)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df if present, else load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# ensure types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# timezone
tz_sh = pytz.timezone('Asia/Shanghai')

# 1) compute global P95 per (cmdb_id, kpi_name) using the full series (thresholds BEFORE window filtering)
metric_global_p95 = metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value'] \
    .quantile(0.95).reset_index(name='global_p95')

# incident window (convert Asia/Shanghai to UTC timestamps)
start_ts = int(pd.Timestamp('2021-03-23 06:30:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 07:00:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# helper to find earliest anomaly for a given cmdb_id and kpi_name
def find_earliest_anomaly(cmdb_id, kpi_name, global_p95_df, window_df):
    row = global_p95_df[(global_p95_df['cmdb_id']==cmdb_id) & (global_p95_df['kpi_name']==kpi_name)]
    if row.empty or pd.isna(row.iloc[0]['global_p95']):
        return None
    gp95 = float(row.iloc[0]['global_p95'])
    # filter window for this series
    series_win = window_df[(window_df['cmdb_id']==cmdb_id) & (window_df['kpi_name']==kpi_name)].copy()
    if series_win.empty:
        return None
    # find earliest timestamp where value >= gp95
    anoms = series_win[series_win['value'] >= gp95]
    if anoms.empty:
        return None
    earliest_ts = int(anoms['timestamp'].min())
    # get the value at earliest timestamp (if multiple rows at same ts, take the first)
    val_at_earliest = float(anoms[anoms['timestamp']==earliest_ts].iloc[0]['value'])
    severity = np.nan if gp95 == 0 else val_at_earliest / gp95
    # format timestamp to Asia/Shanghai
    ts_str = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz_sh).strftime('%Y-%m-%d %H:%M:%S%z')
    return {
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'earliest_anomaly_ts': ts_str,
        'value_at_earliest': round(val_at_earliest,6),
        'global_p95': round(gp95,6),
        'severity_ratio': round(severity,6) if not np.isnan(severity) else None
    }

results = []

# 1-2) MG01 KPI(s) matching pattern 'NETWORK_TCP-FIN-WAIT' OR exact name
mg01_matches = metric_global_p95[
    (metric_global_p95['cmdb_id']=='MG01') &
    (
        metric_global_p95['kpi_name'].str.contains('NETWORK_TCP-FIN-WAIT', na=False) |
        (metric_global_p95['kpi_name']=='OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT')
    )
]['kpi_name'].unique().tolist()

# If multiple matches, pick each (but user expects the TCP-FIN-WAIT KPI)
if not mg01_matches:
    # no matching KPI -> return None record
    results.append({
        'cmdb_id':'MG01',
        'kpi_name': None,
        'earliest_anomaly_ts': None,
        'value_at_earliest': None,
        'global_p95': None,
        'severity_ratio': None
    })
else:
    # take each match; user requested a single KPI record — include all matching (usually one)
    for k in mg01_matches:
        res = find_earliest_anomaly('MG01', k, metric_global_p95, metric_window)
        if res is None:
            results.append({
                'cmdb_id':'MG01',
                'kpi_name': k,
                'earliest_anomaly_ts': None,
                'value_at_earliest': None,
                'global_p95': float(metric_global_p95[(metric_global_p95['cmdb_id']=='MG01') & (metric_global_p95['kpi_name']==k)]['global_p95'].iloc[0]),
                'severity_ratio': None
            })
        else:
            results.append(res)

# 3) MG02, kpi_name 'OSLinux-CPU_CPU-3_SingleCpuidle'
k_mg02 = 'OSLinux-CPU_CPU-3_SingleCpuidle'
res_mg02 = find_earliest_anomaly('MG02', k_mg02, metric_global_p95, metric_window)
if res_mg02 is None:
    # ensure we still return a row with None per instructions
    # attempt to fetch gp95 if exists
    gp_row = metric_global_p95[(metric_global_p95['cmdb_id']=='MG02') & (metric_global_p95['kpi_name']==k_mg02)]
    gp95_val = float(gp_row['global_p95'].iloc[0]) if not gp_row.empty else None
    results.append({
        'cmdb_id':'MG02',
        'kpi_name': k_mg02,
        'earliest_anomaly_ts': None,
        'value_at_earliest': None,
        'global_p95': round(gp95_val,6) if gp95_val is not None else None,
        'severity_ratio': None
    })
else:
    results.append(res_mg02)

# build compact DataFrame with up to two records (if MG01 had multiple matches, include them)
results_df = pd.DataFrame(results)[[
    'cmdb_id','kpi_name','earliest_anomaly_ts','value_at_earliest','global_p95','severity_ratio'
]]

results_df

```
Out[9]:
```
Summary (thresholds = global P95 computed from full series):

- MG01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT:
  - Earliest anomaly: 2021-03-23 06:42:00+0800
  - Value at earliest: 1.0
  - Global P95: 0.5
  - Severity ratio: 2.00 (value is 2× the global P95)

- MG02 — OSLinux-CPU_CPU-3_SingleCpuidle:
  - Earliest anomaly: 2021-03-23 06:49:00+0800
  - Value at earliest: 2.5087
  - Global P95: 0.75261
  - Severity ratio: 3.33 (value is ~3.33× the global P95)

Interpretation: Both KPIs exceeded their historical 95th-percentile during the incident window; MG02 CPU core 3 idle shows the larger relative breach and MG01 TCP-FIN-WAIT doubled its P95. Investigate MG02 CPU/core behavior and MG01 TCP connection state issues.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      kpi_name       earliest_anomaly_ts  value_at_earliest  global_p95  severity_ratio
0    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-23 06:42:00+0800             1.0000     0.50000        2.000000
1    MG02               OSLinux-CPU_CPU-3_SingleCpuidle  2021-03-23 06:49:00+0800             2.5087     0.75261        3.333333```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df and metric_global_p95 if available, else load/compute
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

if 'metric_global_p95' not in globals():
    metric_global_p95 = metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')

# define window in UTC based on Asia/Shanghai (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2021-03-23 06:30:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 07:00:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())

# restrict to components MG01 and MG02 and the time window
candidates = {'MG01','MG02'}
metric_window_candidates = metric_df[
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['cmdb_id'].isin(candidates))
].copy()

# attach global_p95 (computed from full series)
metric_window_candidates = metric_window_candidates.merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')

# mark anomaly rows where value >= global_p95 (and global_p95 present)
metric_window_candidates['is_anomaly'] = metric_window_candidates['global_p95'].notnull() & (metric_window_candidates['value'] >= metric_window_candidates['global_p95'])

anoms = metric_window_candidates[metric_window_candidates['is_anomaly']].copy()

# prepare results per component
records = []
for comp in ['MG01','MG02']:
    comp_anoms = anoms[anoms['cmdb_id']==comp]
    if comp_anoms.empty:
        records.append({
            'component': comp,
            'earliest_anomaly_ts': None,
            'number_of_distinct_kpis_anomalous_at_that_ts': 0,
            'top_kpi_at_that_ts': None,
            'top_kpi_value': None,
            'top_kpi_global_p95': None,
            'top_kpi_severity_ratio': None
        })
        continue

    # find earliest anomaly timestamp for this component
    earliest_ts = int(comp_anoms['timestamp'].min())

    # rows at that timestamp
    at_ts = comp_anoms[comp_anoms['timestamp']==earliest_ts].copy()

    # number of distinct KPIs anomalous at that timestamp
    num_distinct_kpis = int(at_ts['kpi_name'].nunique())

    # compute severity per row safely (avoid divide by zero)
    at_ts['severity'] = np.where(at_ts['global_p95']==0, np.nan, at_ts['value'] / at_ts['global_p95'])

    # aggregate per kpi_name in case multiple rows per KPI at same timestamp: take max value and corresponding severity (based on that max value)
    kpi_agg = at_ts.groupby('kpi_name', dropna=False).agg(
        top_kpi_value=('value','max'),
        top_kpi_global_p95=('global_p95','first')  # same per kpi
    ).reset_index()
    kpi_agg['top_kpi_severity_ratio'] = np.where(kpi_agg['top_kpi_global_p95']==0, np.nan, kpi_agg['top_kpi_value']/kpi_agg['top_kpi_global_p95'])

    # select KPI with largest severity_ratio (tie-breaker: largest value)
    kpi_agg = kpi_agg.sort_values(['top_kpi_severity_ratio','top_kpi_value'], ascending=[False, False]).reset_index(drop=True)
    top = kpi_agg.iloc[0]

    # format earliest timestamp to Asia/Shanghai
    ts_str = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz_sh).strftime('%Y-%m-%d %H:%M:%S%z')

    records.append({
        'component': comp,
        'earliest_anomaly_ts': ts_str,
        'number_of_distinct_kpis_anomalous_at_that_ts': num_distinct_kpis,
        'top_kpi_at_that_ts': top['kpi_name'],
        'top_kpi_value': round(float(top['top_kpi_value']),6),
        'top_kpi_global_p95': round(float(top['top_kpi_global_p95']),6) if pd.notnull(top['top_kpi_global_p95']) else None,
        'top_kpi_severity_ratio': round(float(top['top_kpi_severity_ratio']),6) if pd.notnull(top['top_kpi_severity_ratio']) else None
    })

# final compact DataFrame with exactly two records (MG01 then MG02)
results_df = pd.DataFrame(records)[[
    'component','earliest_anomaly_ts','number_of_distinct_kpis_anomalous_at_that_ts',
    'top_kpi_at_that_ts','top_kpi_value','top_kpi_global_p95','top_kpi_severity_ratio'
]]

results_df

```
Out[10]:
```
Summary (MG01 and MG02 earliest anomaly moments within 2021-03-23 06:30→07:00 UTC+8):

- MG01
  - Earliest anomaly timestamp: 2021-03-23 06:30:00+0800
  - Number of distinct KPIs anomalous at that timestamp: 8
  - Top KPI at that timestamp: OSLinux-CPU_CPU_CPUidleutil
    - Value at that timestamp: 74.477300
    - Global P95 (full-series): 74.425358
    - Severity ratio: 1.000698 (≈0.07% above P95 — a marginal exceedance)

- MG02
  - Earliest anomaly timestamp: 2021-03-23 06:30:00+0800
  - Number of distinct KPIs anomalous at that timestamp: 14
  - Top KPI at that timestamp: OSLinux-CPU_CPU_CPUidleutil
    - Value at that timestamp: 74.412999
    - Global P95 (full-series): 74.332400
    - Severity ratio: 1.001084 (≈0.11% above P95 — a marginal exceedance)

Interpretation: Both components show their earliest anomalies at the start of the window (06:30). The top KPI in both cases is CPU idle utilization and it only slightly exceeds its historical P95 (very small severity ratios), though multiple KPIs are flagged simultaneously (8 and 14), so a coordinated but marginal elevation across metrics is present.

The original code execution output of IPython Kernel is also provided below for reference:

component       earliest_anomaly_ts  number_of_distinct_kpis_anomalous_at_that_ts           top_kpi_at_that_ts  top_kpi_value  top_kpi_global_p95  top_kpi_severity_ratio
0      MG01  2021-03-23 06:30:00+0800                                             8  OSLinux-CPU_CPU_CPUidleutil      74.477300           74.425358                1.000698
1      MG02  2021-03-23 06:30:00+0800                                            14  OSLinux-CPU_CPU_CPUidleutil      74.412999           74.332400                1.001084```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df and metric_global_p95 if available, else load/compute
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

if 'metric_global_p95' not in globals():
    metric_global_p95 = metric_df.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')

# timezone and window (Asia/Shanghai -> convert to UTC timestamps)
tz_sh = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2021-03-23 06:30:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())
end_ts   = int(pd.Timestamp('2021-03-23 07:00:00', tz=tz_sh).astimezone(pytz.UTC).timestamp())

# target KPIs
targets = [
    ('MG01', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT'),
    ('MG02', 'OSLinux-CPU_CPU-3_SingleCpuidle')
]

rows = []
for cmdb_id, kpi in targets:
    # get global p95 for this KPI (from full series)
    gp_row = metric_global_p95[(metric_global_p95['cmdb_id']==cmdb_id) & (metric_global_p95['kpi_name']==kpi)]
    gp95 = float(gp_row['global_p95'].iloc[0]) if not gp_row.empty else np.nan

    # filter metric_df to window & this KPI
    series_win = metric_df[
        (metric_df['cmdb_id']==cmdb_id) &
        (metric_df['kpi_name']==kpi) &
        (metric_df['timestamp'] >= start_ts) &
        (metric_df['timestamp'] <= end_ts)
    ].copy().sort_values('timestamp')

    if series_win.empty or np.isnan(gp95):
        # indicate none
        rows.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi,
            'run_start_ts': None,
            'run_end_ts': None,
            'duration_minutes': 0,
            'max_value_in_run': None,
            'global_p95': (round(gp95,6) if not np.isnan(gp95) else None),
            'severity_ratio': None
        })
        continue

    # mark anomaly rows where value >= gp95
    series_win['is_anom'] = series_win['value'] >= gp95

    anom_rows = series_win[series_win['is_anom']].copy()
    if anom_rows.empty:
        rows.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi,
            'run_start_ts': None,
            'run_end_ts': None,
            'duration_minutes': 0,
            'max_value_in_run': None,
            'global_p95': round(gp95,6),
            'severity_ratio': None
        })
        continue

    # identify consecutive-minute runs (diff == 60 sec)
    anom_rows = anom_rows.sort_values('timestamp').reset_index(drop=True)
    anom_rows['ts_diff'] = anom_rows['timestamp'].diff().fillna(999999)
    anom_rows['new_run'] = (anom_rows['ts_diff'] != 60).astype(int)
    anom_rows['run_id'] = anom_rows['new_run'].cumsum()

    runs = anom_rows.groupby('run_id').agg(
        run_start_ts=('timestamp','min'),
        run_end_ts=('timestamp','max'),
        duration_minutes=('timestamp','count'),
        max_value_in_run=('value','max')
    ).reset_index(drop=True)

    # attach global_p95 and severity ratio
    runs['global_p95'] = round(gp95,6)
    runs['severity_ratio'] = runs['max_value_in_run'] / gp95
    runs['max_value_in_run'] = runs['max_value_in_run'].round(6)
    runs['severity_ratio'] = runs['severity_ratio'].round(6)

    # convert timestamps to Asia/Shanghai strings
    runs['run_start_ts'] = pd.to_datetime(runs['run_start_ts'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    runs['run_end_ts'] = pd.to_datetime(runs['run_end_ts'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # append rows for each run
    for _, r in runs.iterrows():
        rows.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi,
            'run_start_ts': r['run_start_ts'],
            'run_end_ts': r['run_end_ts'],
            'duration_minutes': int(r['duration_minutes']),
            'max_value_in_run': r['max_value_in_run'],
            'global_p95': r['global_p95'],
            'severity_ratio': r['severity_ratio']
        })

# final compact DataFrame (all runs for the two KPIs; if none, shows None rows)
runs_df = pd.DataFrame(rows)[[
    'cmdb_id','kpi_name','run_start_ts','run_end_ts','duration_minutes','max_value_in_run','global_p95','severity_ratio'
]]

runs_df

```
Out[11]:
```
Summary of detected runs (value >= global P95) within 2021-03-23 06:30 → 07:00 UTC+8:

- MG01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
  - Run 1: 2021-03-23 06:42:00+0800 → 2021-03-23 06:42:00+0800
    - duration: 1 minute
    - max_value_in_run: 1.0
    - global_p95: 0.5
    - severity_ratio: 2.00
  - Run 2: 2021-03-23 06:46:00+0800 → 2021-03-23 06:46:00+0800
    - duration: 1 minute
    - max_value_in_run: 2.0
    - global_p95: 0.5
    - severity_ratio: 4.00

- MG02 — OSLinux-CPU_CPU-3_SingleCpuidle
  - Run: 2021-03-23 06:49:00+0800 → 2021-03-23 06:49:00+0800
    - duration: 1 minute
    - max_value_in_run: 2.5087
    - global_p95: 0.75261
    - severity_ratio: 3.333333

Note: All detected runs are single-minute events (no multi-minute consecutive runs found for these two KPIs in the window).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      kpi_name              run_start_ts                run_end_ts  duration_minutes  max_value_in_run  global_p95  severity_ratio
0    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-23 06:42:00+0800  2021-03-23 06:42:00+0800                 1            1.0000     0.50000        2.000000
1    MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-23 06:46:00+0800  2021-03-23 06:46:00+0800                 1            2.0000     0.50000        4.000000
2    MG02               OSLinux-CPU_CPU-3_SingleCpuidle  2021-03-23 06:49:00+0800  2021-03-23 06:49:00+0800                 1            2.5087     0.75261        3.333333```
```